<a href="https://colab.research.google.com/github/prathyusha2020/AAIS_322-Natural-Language-Processing-and-Computer-Vision/blob/main/Module_2_Interactive_Session_1_07_29_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Natural Language Processing and Computer Vision

##Module 2: Exploring Sequence-to-Sequence Models in NLP

###Topics covered:

1. spaCy Overview
2. Example Activity for Matching
3. Using PhraseMatcher
4. EntityRuler in spaCy
5. Hugging Face
6. Loading squence to sequence models
7. Loading opensource models

In [ ]:
!pip install -q -U spacy
!python -m spacy download en_core_web_sm

In [21]:
import spacy

In [22]:
nlp = spacy.load("en_core_web_sm")

In [23]:
# What did we actually just load? Look inside the pipeline.
print("Pipeline components:", nlp.pipe_names)
print()
print("Language     :", nlp.meta["lang"])
print("Model name   :", nlp.meta["name"])
print("Version      :", nlp.meta["version"])
print("Has vectors  :", nlp.vocab.vectors.shape)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

Language     : en
Model name   : core_web_sm
Version      : 3.8.0
Has vectors  : (0, 0)


> ### ❓ Check your understanding
>
> **Q1.** The tokenizer is not in `nlp.pipe_names`. Does that mean the text isn't tokenized?
>
> <details><summary>Answer</summary>
>
> No. The tokenizer is special — it always runs first and is not optional, so spaCy keeps
> it outside the configurable pipeline list. Everything in `pipe_names` operates *on
> tokens*, which means tokenization must already have happened.
> </details>
>
> **Q2.** You only need named entities from a 500,000-document corpus. Which components
> should you disable, and why does it matter?
>
> <details><summary>Answer</summary>
>
> Disable `parser` and `lemmatizer` — NER doesn't depend on them. Keep `tok2vec` and
> `tagger`, since `ner` uses `tok2vec`'s representations. The parser is the most
> expensive component in the pipeline, so this can cut runtime by more than half. On
> half a million documents that's the difference between minutes and an hour.
> </details>

In [24]:
doc = nlp("Apple is looking to buying UK startup for $1 billion dollars.")

In [26]:
doc[0] - # first token
print(len(doc))

13


In [16]:
print("Whole doc:", doc.text)

Whole doc: Google is opening a new office in Berlin on July 15, 2025. Azam is hiking on Rocky Mountain Colorado. Estes Park, CO just did a renovation for 20 million.


In [27]:

print("Tokens: ", [token.text for token in doc])

Tokens:  ['Apple', 'is', 'looking', 'to', 'buying', 'UK', 'startup', 'for', '$', '1', 'billion', 'dollars', '.']


In [28]:
# One token has many attributes, not just .text
import pandas as pd

rows = []
for token in doc:
    rows.append({
        "text":     token.text,      # surface form as written
        "lemma_":   token.lemma_,    # dictionary form: "buying" -> "buy"
        "pos_":     token.pos_,      # coarse universal POS tag: NOUN, VERB, PROPN
        "tag_":     token.tag_,      # fine-grained Penn Treebank tag: NN, VBG, NNP
        "dep_":     token.dep_,      # syntactic role relative to its head
        "head":     token.head.text, # the word this token attaches to
        "is_stop":  token.is_stop,   # is it a very common function word?
        "is_alpha": token.is_alpha,  # letters only?
        "is_punct": token.is_punct,  # punctuation?
        "like_num": token.like_num,  # looks numeric? ("1", "one", "10.5")
        "idx":      token.idx,       # character offset in the original string
    })

pd.DataFrame(rows)

,text,lemma_,pos_,tag_,dep_,head,is_stop,is_alpha,is_punct,like_num,idx
0,Apple,Apple,PROPN,NNP,nsubj,looking,False,True,False,False,0
1,is,be,AUX,VBZ,aux,looking,True,True,False,False,6
2,looking,look,VERB,VBG,ROOT,looking,False,True,False,False,9
3,to,to,ADP,IN,prep,looking,True,True,False,False,17
4,buying,buy,VERB,VBG,pcomp,to,False,True,False,False,20
5,UK,UK,PROPN,NNP,dobj,buying,False,True,False,False,27
6,startup,startup,VERB,VBD,advcl,looking,False,True,False,False,30
7,for,for,ADP,IN,prep,startup,True,True,False,False,38
8,$,$,SYM,$,quantmod,billion,False,False,False,False,42
9,1,1,NUM,CD,compound,billion,False,False,False,True,43


In [29]:
doc = nlp("Google is opening a new office in Berlin on July 15, 2025. Azam is hiking on Rocky Mountain Colorado. Estes Park, CO just did a renovation for 20 million.")

for ent in doc.ents:
  print(ent.text, ent.label_)

Google ORG
Berlin GPE
July 15, 2025 DATE
Azam PERSON
Rocky Mountain Colorado LOC
Estes Park GPE
20 million CARDINAL


In [32]:
spacy.explain("Google is opening a new office in Berlin on July 15, 2025. Azam is hiking on Rocky Mountain Colorado. Estes Park, CO just did a renovation for 20 million.")


/usr/local/lib/python3.12/dist-packages/spacy/glossary.py:20: UserWarning: [W118] Term 'Google is opening a new office in Berlin on July 15, 2025. Azam is hiking on Rocky Mountain Colorado. Estes Park, CO just did a renovation for 20 million.' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


In [13]:

from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

## Part 1b · Named Entity Recognition (NER)

In [33]:
doc = nlp("Google is opening a new office in Berlin on July 15, 2025. Azam is hiking on Rocky Mountain Colorado. Estes Park, CO just did a renovation for 20 million.")

for ent in doc.ents:
  print(ent.text, ent.label_)

Google ORG
Berlin GPE
July 15, 2025 DATE
Azam PERSON
Rocky Mountain Colorado LOC
Estes Park GPE
20 million CARDINAL


In [35]:
spacy.explain("GPE")

'Countries, cities, states'

In [52]:
# Never memorise label names - ask spaCy
for label in ["ORG", "GPE", "LOC", "DATE", "MONEY", "CARDINAL", "NORP", "FAC", "PRODUCT"]:
    print(f"{label:10s} -> {spacy.explain(label)}")

print()
print("It works on POS and dependency tags too:")
for tag in ["PROPN", "VBG", "nsubj", "pobj", "compound"]:
    print(f"{tag:10s} -> {spacy.explain(tag)}")



ORG        -> Companies, agencies, institutions, etc.
GPE        -> Countries, cities, states
LOC        -> Non-GPE locations, mountain ranges, bodies of water
DATE       -> Absolute or relative dates or periods
MONEY      -> Monetary values, including unit
CARDINAL   -> Numerals that do not fall under another type
NORP       -> Nationalities or religious or political groups
FAC        -> Buildings, airports, highways, bridges, etc.
PRODUCT    -> Objects, vehicles, foods, etc. (not services)

It works on POS and dependency tags too:
PROPN      -> proper noun
VBG        -> verb, gerund or present participle
nsubj      -> nominal subject
pobj       -> object of preposition
compound   -> compound


In [53]:
# Full detail on every entity found, including where it sits in the string
rows = []
for ent in doc.ents:
    rows.append({
        "text":       ent.text,
        "label_":     ent.label_,
        "explained":  spacy.explain(ent.label_),
        "start_char": ent.start_char,
        "end_char":   ent.end_char,
        "n_tokens":   len(ent),
    })

pd.DataFrame(rows)

""


In [54]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

/usr/local/lib/python3.12/dist-packages/spacy/displacy/__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [50]:
from spacy import displacy
# The dependency parse of a short sentence - what the `parser` component produced
short = nlp("Apple bought a UK startup for $1 billion.")
displacy.render(short, style="dep", jupyter=True, options={"distance": 110})

**What is happening:** we render the *syntactic* structure — which word depends on which.
`nsubj` is the subject, `dobj` the direct object, `prep` a preposition, `pobj` its object.

**Why show this in a Module 2 class:** it makes the "pipeline" claim concrete. That tree
was built during the *same* `nlp()` call that produced the tokens and the entities. One
pass, many layers of annotation. It also sets up an honest comparison for later: this is
what *classical, structured* NLP gives you. A large language model gives you fluent text
but no explicit tree like this. Different tools, different jobs.

---
> ### Check your understanding
>
> **Q3.** `doc.ents` returned `20 million` as `CARDINAL`, but `$1 billion` earlier was
> `MONEY`. Is this a bug?
>
> <details><summary>Answer</summary>
>
> No — it's the model behaving as trained. It learned that a currency symbol or currency
> word is the signal for `MONEY`. "20 million" on its own genuinely is ambiguous; it could
> be 20 million users or 20 million tons. The lesson: NER predicts from surface evidence,
> not from your intent. If your domain always means dollars, you encode that yourself with
> an `EntityRuler`.
> </details>
>
> **Q4.** What is the difference between `ent.start` and `ent.start_char`?
>
> <details><summary>Answer</summary>
>
> `ent.start` is the index of the entity's first **token** within the `Doc`.
> `ent.start_char` is the index of its first **character** within the original string.
> You use token indices to build a new `Span` in spaCy; you use character offsets to
> highlight the text in a webpage or store the location in a database.
> </details>
>
> **Q5.** Your resume parser must always tag `SwiftUI` as a technology. NER misses it.
> What are your two options, and which is realistic?
>
> <details><summary>Answer</summary>
>
> Option 1: annotate thousands of resumes and retrain the NER component — expensive, slow,
> and overkill for a fixed vocabulary. Option 2: add a rule. `SwiftUI` is a known, finite
> string; you don't need statistics to find a known string. Rules are the right tool
> whenever your target list is finite and enumerable. Statistical NER is for the
> open-ended cases (any person's name, any company).
> </details>

In [59]:
doc=nlp("What is the difference between Windows and Apple?")

print(f"Number of entities found: {len(doc.ents)}")
print(f"Entities: {doc.ents}")

if len(doc.ents) > 0:

    ent=doc.ents[0]
    print("\nDetails for the first entity:")
    print(ent)
    print(ent.text, ent.start, ent.start_char, ent.end, ent.end_char)
    print(doc[ent.start:ent.end])

else:
    print("\nNo entities were detected in the sentence.")

Number of entities found: 2
Entities: (Windows, Apple)

Details for the first entity:
Windows
Windows 5 31 6 38
Windows


In [55]:
print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [58]:
print(list(doc.ents))

[]


In [56]:
print(ent.text)
print(ent.start, ent.end)
print(ent.start_char, ent.end_char)

ent.start
5 6
31 40


# Rule-Based Matching

##matching based on tokens

In [73]:
from spacy.matcher import Matcher
from spacy.util import filter_spans

nlp = spacy.load("en_core_web_sm")          # fresh pipeline, no rules attached
matcher = Matcher(nlp.vocab)

# A pattern is a LIST OF DICTS - one dict per token, in order.
# This one means: a "$" symbol, then a number, then optionally a scale word.
money_pattern = [
    {"ORTH": "$"},                                       # exactly the character "$"
    {"LIKE_NUM": True},                                   # any token that looks numeric
    {"LOWER": {"IN": ["million", "billion", "thousand"]}, "OP": "?"},   # optional
]
matcher.add("MONEY_AMOUNT", [money_pattern])

# A course code like AAIS-322. NOTE: spaCy keeps "AAIS-322" as ONE token,


text = "AAIS costs $1 billion, CS-101 costs $40 thousand, and lunch costs $12."
doc = nlp(text)
course_pattern = [
    {"TEXT": {"REGEX": r"^[A-Z]{2,4}-\d{2,4}$"}},
]
matcher.add("COURSE_CODE", [course_pattern])
matches = matcher(doc)
print("Raw matches (match_id, start, end):", matches)
print()

for match_id, start, end in matches:
    rule_name = nlp.vocab.strings[match_id]      # int hash -> the name you registered
    span = doc[start:end]                        # slice the Doc to get a Span
    print(f"{rule_name:14s} | tokens {start}:{end} | '{span.text}'")

Raw matches (match_id, start, end): [(7081236657488100519, 2, 4), (7081236657488100519, 2, 5), (16910664930825921930, 6, 7), (7081236657488100519, 8, 10), (7081236657488100519, 8, 11), (7081236657488100519, 15, 17)]

MONEY_AMOUNT   | tokens 2:4 | '$1'
MONEY_AMOUNT   | tokens 2:5 | '$1 billion'
COURSE_CODE    | tokens 6:7 | 'CS-101'
MONEY_AMOUNT   | tokens 8:10 | '$40'
MONEY_AMOUNT   | tokens 8:11 | '$40 thousand'
MONEY_AMOUNT   | tokens 15:17 | '$12'


##EntityRuler

In [76]:
import spacy
from spacy.pipeline import EntityRuler

nlp = spacy.load("en_core_web_sm")

# Create EntityRuler and add to pipeline BEFORE ner
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add patterns (you can assign them to any label you want)
patterns = [
    {"label": "TECH", "pattern": "scikit-learn"},
    {"label": "TECH", "pattern": "SwiftUI"},
    {"label": "ORG", "pattern": "AzamSharp School"},
    {"label": "ORG", "pattern": "learning"}
]
ruler.add_patterns(patterns)

doc = nlp("I am learning SwiftUI and scikit-learn with AzamSharp School.")

for ent in doc.ents:
    print(ent.text, ent.label_)

learning ORG
SwiftUI TECH
scikit-learn TECH
AzamSharp School ORG


##PhraseMatcher

In [77]:
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy.util import filter_spans

nlp = spacy.load("en_core_web_sm")

# Initialize the PhraseMatcher
phrase_matcher = PhraseMatcher(nlp.vocab)

# List of phrases to look for
phrases = ["machine learning", "artificial intelligence", "natural language processing"]

# Convert phrases into spaCy docs
patterns = [nlp.make_doc(p) for p in phrases]

phrase_matcher.add("AI_TERMS", patterns)

doc = nlp("This course teaches machine learning and natural language processing.")

matches = phrase_matcher(doc)
for match_id, start, end in matches:
    print("Matched:", doc[start:end].text)

Matched: machine learning
Matched: natural language processing


In [78]:
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy.util import filter_spans

nlp = spacy.load("en_core_web_sm")

matcher = PhraseMatcher(nlp.vocab)
terms = ["FastAPI", "PyTorch", "Core ML"]
patterns = [nlp.make_doc(t) for t in terms]
matcher.add("TECH", patterns)

doc = nlp("We used FastAPI and Core ML for deployment.")

matches = matcher(doc)
new_ents = []
for match_id, start, end in matches:
    span = Span(doc, start, end, label="TECH")
    new_ents.append(span)

# Combine new_ents (which are our rule-based matches) with existing doc.ents
# Order matters: new_ents first ensures they take precedence in case of overlap
combined_ents = new_ents + list(doc.ents)

# Filter out overlapping spans, preferring the longer ones, or the ones that came first in the list (our new_ents)
doc.ents = filter_spans(combined_ents)

for ent in doc.ents:
    print(ent.text, ent.label_)

FastAPI TECH
Core ML TECH


**What is happening in this cell**

One `EntityRuler` holding two *kinds* of pattern:

- **String patterns** — `"scikit-learn"`, `"SwiftUI"`. Exact phrase lookup. Fast, literal.
- **Token patterns** — the `COURSE` and `VERSION` rules. These match *structure*, so they
  catch `AAIS-322` **and** `CS-101` without either being listed. You never enumerated
  course codes; you described their shape.

That contrast is the heart of the whole matching section:

> **String pattern = "I know exactly what I'm looking for."**
> **Token pattern = "I know what it looks like, not what it says."**

**The `id` field:** both `sklearn` and `scikit-learn` carry `id="sklearn"`. Read it back
with `ent.ent_id_`. This is entity *normalisation* — three spellings, one canonical
concept. In production this is how you avoid counting `NYC`, `New York City`, and `New
York, NY` as three different places.

---
> ### Check your understanding
>
> **Q6.** Why `nlp.make_doc(term)` instead of `nlp(term)` when building `PhraseMatcher`
> patterns?
>
> <details><summary>Answer</summary>
>
> `make_doc` runs only the tokenizer. `nlp()` runs the full pipeline — tagger, parser,
> lemmatizer, NER — on each term, which is completely wasted work since matching only needs
> tokens. On a list of 50,000 terms this is the difference between under a second and
> several minutes. Identical results, orders of magnitude apart in cost.
> </details>
>
> **Q7.** You need to find every mention of *"buy"* including "bought", "buying", "buys".
> Matcher, PhraseMatcher, or EntityRuler — and what's the pattern?
>
> <details><summary>Answer</summary>
>
> `Matcher` (or an `EntityRuler` token pattern) with `[{"LEMMA": "buy"}]`. `PhraseMatcher`
> is the wrong tool — it matches literal strings, so you'd have to enumerate every
> inflected form by hand. Matching on `LEMMA` lets the lemmatizer do that work for you,
> which is why the lemmatizer has to run before your rule.
> </details>
>
> **Q8.** Both your `EntityRuler` and spaCy's `ner` want to label the same words. Who wins,
> and how do you control it?
>
> <details><summary>Answer</summary>
>
> Whoever runs first in the pipeline. Control it with the `before=` / `after=` argument to
> `add_pipe`. `before="ner"` means your rules win. `after="ner"` means the statistical model
> wins and your overlapping rules are silently dropped — no exception, no warning.
> </details>
>
> **Q9.** When would you deliberately choose statistical NER over rules?
>
> <details><summary>Answer</summary>
>
> When the target set is open-ended. You cannot enumerate every human name, every company,
> every city — new ones appear constantly. Rules handle closed vocabularies; statistics
> handle open ones. Real systems use both: `EntityRuler` for your known product catalogue,
> `ner` for everything unpredictable.
> </details>